In [ ]:
import requests
import random
import json
import time
from datetime import datetime

# =========================
# CONFIG
# =========================

OUTPUT_SQL_FILE = "book_seed.sql"
TOTAL_RECORDS = 10000

SUBJECTS = [
    "fantasy",
    "science_fiction",
    "romance",
    "history",
    "mystery",
    "thriller",
    "horror",
    "biography",
    "technology",
    "programming",
    "philosophy",
    "psychology",
    "business",
    "economics",
    "politics",
    "self_help",
    "fiction",
    "young_adult",
    "children",
    "adventure"
]

LANGUAGES = ["en", "es", "fr", "de", "it", "ja"]

EDITIONS = [
    "1st Edition",
    "2nd Edition",
    "Revised Edition",
    "Collector's Edition",
    "Paperback",
    "Hardcover"
]

GENRES = [
    "Fantasy",
    "Sci-Fi",
    "Romance",
    "Mystery",
    "Thriller",
    "Programming",
    "Business",
    "Biography",
    "Psychology",
    "History",
    "Adventure",
    "Horror",
    "Young Adult"
]

PUBLISHERS = [
    "Penguin Random House",
    "HarperCollins",
    "Simon & Schuster",
    "O'Reilly Media",
    "Macmillan",
    "Oxford Press",
    "Scholastic",
    "Vintage Books",
    "No Starch Press"
]

MAX_DESCRIPTION_LENGTH = 5000
SUBJECT_PAGE_LIMIT = 100

# =========================
# HELPERS
# =========================

session = requests.Session()

used_isbn13 = set()
used_isbn10 = set()

work_cache = {}


def sanitize(v):
    if v is None:
        return ""
    return str(v).replace("'", "''").replace("\x00", "")


def safe_json(v):
    return sanitize(
        json.dumps(
            v,
            ensure_ascii=False
        )
    )


def random_price():
    return round(
        random.uniform(5.99, 49.99),
        2
    )


def build_metadata(subject):
    return {
        "genre": random.choice(GENRES),
        "subject": subject,
        "price": random_price(),
        "rating": round(
            random.uniform(3.2, 5.0),
            1
        ),
        "in_stock": random.choice(
            [True, False]
        ),
        "format": random.choice(
            [
                "Paperback",
                "Hardcover",
                "eBook"
            ]
        ),
        "tags": random.sample(
            [
                "popular",
                "classic",
                "award-winning",
                "bestseller",
                "recommended",
                "editor-pick",
                "fiction",
                "non-fiction"
            ],
            3
        )
    }


def generate_isbn13():
    while True:
        isbn = (
            "978"
            + str(
                random.randint(
                    1000000000,
                    9999999999
                )
            )
        )[:13]

        if isbn not in used_isbn13:
            used_isbn13.add(isbn)
            return isbn


def generate_isbn10():
    while True:
        isbn = str(
            random.randint(
                1000000000,
                9999999999
            )
        )

        if isbn not in used_isbn10:
            used_isbn10.add(isbn)
            return isbn


def fetch_work(work_key):

    if not work_key:
        return {}

    if work_key in work_cache:
        return work_cache[work_key]

    try:
        r = session.get(
            f"https://openlibrary.org{work_key}.json",
            timeout=20
        )

        r.raise_for_status()

        data = r.json()

        work_cache[work_key] = data

        time.sleep(0.05)

        return data

    except Exception:
        return {}


def extract_description(
    work,
    title,
    subject
):

    desc = work.get(
        "description"
    )

    if isinstance(
        desc,
        dict
    ):
        desc = desc.get(
            "value"
        )

    if isinstance(
        desc,
        str
    ):
        desc = desc.strip()

    if desc:
        return sanitize(
            desc[
                :MAX_DESCRIPTION_LENGTH
            ]
        )

    excerpts = work.get(
        "excerpts",
        []
    )

    if excerpts:

        e = excerpts[0]

        if isinstance(
            e,
            dict
        ):
            txt = e.get(
                "excerpt"
            )

            if isinstance(
                txt,
                dict
            ):
                txt = txt.get(
                    "value"
                )

            if txt:
                return sanitize(
                    txt[
                        :MAX_DESCRIPTION_LENGTH
                    ]
                )

    subjects = work.get(
        "subjects",
        []
    )

    if subjects:

        return sanitize(
            (
                f"{title}. "
                f"This book explores: "
                + ", ".join(
                    subjects[:15]
                )
            )
        )

    return sanitize(
        f"{title}. "
        f"No description available."
    )


# =========================
# GENERATION
# =========================

generated = 0

with open(
    OUTPUT_SQL_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "-- Auto generated seed\n"
    )

    f.write(
        f"-- Generated {datetime.utcnow().isoformat()}Z\n\n"
    )

    for subject in SUBJECTS:

        if generated >= TOTAL_RECORDS:
            break

        print(
            f"Loading {subject}"
        )

        offset = 0

        while generated < TOTAL_RECORDS:

            try:

                url = (
                    "https://openlibrary.org"
                    f"/subjects/{subject}.json"
                    f"?limit={SUBJECT_PAGE_LIMIT}"
                    f"&offset={offset}"
                )

                r = session.get(
                    url,
                    timeout=30
                )

                r.raise_for_status()

                data = r.json()

            except Exception as e:
                print(e)
                break

            works = data.get(
                "works",
                []
            )

            if not works:
                break

            for work in works:

                if generated >= TOTAL_RECORDS:
                    break

                title = sanitize(
                    work.get(
                        "title",
                        "Unknown"
                    )
                )

                work_details = fetch_work(
                    work.get("key")
                )

                authors = [
                    sanitize(
                        a.get(
                            "name"
                        )
                    )
                    for a in work.get(
                        "authors",
                        []
                    )
                    if a.get(
                        "name"
                    )
                ]

                if not authors:
                    authors = [
                        "Unknown Author"
                    ]

                description = (
                    extract_description(
                        work_details,
                        title,
                        subject
                    )
                )

                publish = (
                    work_details.get(
                        "first_publish_date"
                    )
                    or work.get(
                        "first_publish_year"
                    )
                )

                try:
                    year = int(
                        str(
                            publish
                        )[:4]
                    )
                except:
                    year = random.randint(
                        1950,
                        2025
                    )

                publication_date = (
                    f"{year}-"
                    f"{random.randint(1,12):02d}-"
                    f"{random.randint(1,28):02d}"
                )

                page_count = random.randint(
                    80,
                    1200
                )

                cover = (
                    f"https://covers.openlibrary.org"
                    f"/b/id/"
                    f"{work.get('cover_id')}"
                    f"-L.jpg"
                    if work.get(
                        "cover_id"
                    )
                    else None
                )

                subtitle = (
                    random.choice(
                        [
                            "A Novel",
                            "Complete Edition",
                            "Modern Edition",
                            None
                        ]
                    )
                )

                sql = f"""
INSERT INTO book (
isbn_13,
isbn_10,
title,
subtitle,
authors,
publisher,
description,
language_code,
page_count,
publication_date,
edition,
metadata,
cover_image_url
)
VALUES (
'{generate_isbn13()}',
'{generate_isbn10()}',
'{title}',
{"NULL" if not subtitle else "'" + sanitize(subtitle) + "'"},
ARRAY[{",".join("'" + a + "'" for a in authors)}],
'{sanitize(random.choice(PUBLISHERS))}',
'{description}',
'{random.choice(LANGUAGES)}',
{page_count},
'{publication_date}',
'{sanitize(random.choice(EDITIONS))}',
'{safe_json(build_metadata(subject))}'::jsonb,
{"NULL" if not cover else "'" + sanitize(cover) + "'"}
);

"""

                f.write(sql)

                generated += 1

                if generated % 100 == 0:
                    print(
                        f"{generated} rows"
                    )

            offset += SUBJECT_PAGE_LIMIT

print()
print(
    f"Done → {OUTPUT_SQL_FILE}"
)
print(
    f"Generated {generated} rows"
)


Fetching subject: fantasy
Fetching subject: science_fiction
Fetching subject: romance
Fetching subject: history
Generated 1000 INSERT statements
Output file: book_seed.sql


/tmp/ipykernel_5598/3603301778.py:257: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f.write(f"-- Generated at: {datetime.utcnow().isoformat()}Z\n\n")
